# Week 5 — Samsung Galaxy RAG Assistant

RAG over Samsung Galaxy mobile devices. Ask about S24, Z Fold/Flip, A series, and features.

**Models:** OpenAI, Anthropic (ANTHROPIC_API_KEY), or Ollama (local). Embeddings use OpenAI. Gradio UI.

In [ ]:
# Standard library: paths and env
import os
from pathlib import Path
# Load env vars from .env (e.g. OPENAI_API_KEY, ANTHROPIC_API_KEY)
from dotenv import load_dotenv
# OpenAI client for chat and (via Chroma) for embeddings
from openai import OpenAI
# Chroma: in-memory vector store for RAG retrieval
import chromadb
from chromadb.utils import embedding_functions
# Anthropic SDK for Claude (alternative to OpenAI chat)
import anthropic
# Gradio: chat UI
import gradio as gr

In [ ]:
# Load .env so we can read API keys
load_dotenv(override=True)
openai_key = os.getenv("OPENAI_API_KEY")
anthropic_key = os.getenv("ANTHROPIC_API_KEY")

if openai_key:
    print(f"OpenAI key found (starts with {openai_key[:8]}...)")
else:
    print("Set OPENAI_API_KEY in .env (used for embeddings and OpenAI chat)")
if anthropic_key:
    print(f"Anthropic key found (starts with {anthropic_key[:7]}...)")
else:
    print("ANTHROPIC_API_KEY not set (optional; needed for Claude model)")

# OpenAI: used for embeddings and for chat when user selects OpenAI
openai_client = OpenAI()
MODEL_OPENAI = "gpt-4o-mini"

# Anthropic: used for chat when user selects Claude
anthropic_client = anthropic.Anthropic()
MODEL_ANTHROPIC = "claude-3-5-sonnet-20241022"

# Ollama: local server; run "ollama run llama3.2" first. Used for chat when user selects Ollama
ollama_client = OpenAI(api_key="ollama", base_url="http://localhost:11434/v1")
MODEL_OLLAMA = "llama3.2"

print("Clients ready.")

In [ ]:
# Build the knowledge base: read every .md file under knowledge_base/
# Each doc becomes a dict with "title" (from filename) and "content" (file body)
# Run the notebook from this folder so "knowledge_base" is found, or we fall back to cwd/knowledge_base
documents = []
kb_path = Path("knowledge_base")
if not kb_path.exists():
    kb_path = Path.cwd() / "knowledge_base"
for path in sorted(kb_path.glob("*.md")):
    with open(path, "r", encoding="utf-8") as f:
        documents.append({"title": path.stem.replace("_", " ").title(), "content": f.read()})
print(f"Loaded {len(documents)} documents from knowledge_base.")

In [ ]:
# Split each document into smaller chunks (by word count) so we can retrieve
# only the relevant parts for each question. Each chunk keeps a "source" (doc title) for citations.
def chunk_doc(doc, chunk_size=220):
    words = doc["content"].split()
    chunks = []
    for i in range(0, len(words), chunk_size):
        text = " ".join(words[i : i + chunk_size])
        chunks.append({"text": text, "source": doc["title"]})
    return chunks

all_chunks = []
for doc in documents:
    all_chunks.extend(chunk_doc(doc))
print(f"Created {len(all_chunks)} chunks.")

In [ ]:
# Embedding function: turns text into vectors via OpenAI (needed for semantic search).
# We use the same embeddings for indexing and for querying.
openai_ef = embedding_functions.OpenAIEmbeddingFunction(
    api_key=openai_key,
    model_name="text-embedding-3-small"
)
# In-memory Chroma client and a single collection for our Galaxy docs.
chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection(
    name="samsung_galaxy",
    embedding_function=openai_ef,
    metadata={"description": "Samsung Galaxy knowledge base"}
)
# Index all chunks: Chroma will embed each chunk and store it for similarity search.
collection.add(
    documents=[c["text"] for c in all_chunks],
    metadatas=[{"source": c["source"]} for c in all_chunks],
    ids=[f"chunk_{i}" for i in range(len(all_chunks))]
)
print(f"Indexed {collection.count()} chunks in Chroma.")

In [ ]:
# Instructs the LLM to answer only from the retrieved context and to say when it doesn't know.
SYSTEM_PROMPT = """You are a helpful Samsung Galaxy expert. Answer only from the context below about Samsung Galaxy phones (S24, Z Fold, Z Flip, A series, features, software). Be brief and accurate. If the context does not contain the answer, say so."""
RETRIEVAL_K = 4

# Retrieve the top-k most relevant chunks for the question (semantic search via Chroma).
def query_rag(question, n_results=RETRIEVAL_K):
    results = collection.query(query_texts=[question], n_results=n_results)
    return results["documents"][0], results["metadatas"][0]

# Full RAG pipeline: retrieve context, then call the selected model (OpenAI, Anthropic, or Ollama).
def ask_galaxy(question, model_choice):
    if not question or not question.strip():
        return "Ask a question about Samsung Galaxy devices.", ""
    docs, metas = query_rag(question.strip())
    context = "\n\n".join([f"[{m['source']}]: {d}" for d, m in zip(docs, metas)])
    user_msg = f"CONTEXT:\n{context}\n\nQUESTION: {question.strip()}"
    try:
        if model_choice == "OpenAI (gpt-4o-mini)":
            r = openai_client.chat.completions.create(
                model=MODEL_OPENAI,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": user_msg}
                ],
                temperature=0.2
            )
            answer = (r.choices[0].message.content or "").strip()
        elif model_choice == "Anthropic (claude-3-5-sonnet)":
            msg = anthropic_client.messages.create(
                model=MODEL_ANTHROPIC,
                max_tokens=1024,
                system=SYSTEM_PROMPT,
                messages=[{"role": "user", "content": user_msg}],
            )
            answer = (msg.content[0].text if msg.content else "").strip()
        else:
            # Ollama (local)
            r = ollama_client.chat.completions.create(
                model=MODEL_OLLAMA,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": user_msg}
                ],
                temperature=0.2
            )
            answer = (r.choices[0].message.content or "").strip()
        sources = ", ".join(set(m["source"] for m in metas))
        return answer, f"Sources: {sources}"
    except Exception as e:
        return f"Error: {e}", ""

In [ ]:
# Chat handler: call RAG with the user message and selected model; return updated history and clear the input.
def chat_fn(message, history, model_choice):
    answer, sources = ask_galaxy(message, model_choice)
    reply = answer + ("\n\n" + sources if sources else "")
    return history + [[message, reply]], ""

# Gradio UI: model dropdown, chatbot, textbox + submit. Submitting runs RAG with the chosen model.
with gr.Blocks(title="Samsung Galaxy RAG") as demo:
    gr.Markdown("## Samsung Galaxy RAG Assistant\nAsk about S24, Z Fold/Flip, A series, features.")
    model_dropdown = gr.Dropdown(
        choices=["OpenAI (gpt-4o-mini)", "Anthropic (claude-3-5-sonnet)", "Ollama (llama3.2)"],
        value="OpenAI (gpt-4o-mini)",
        label="Model",
    )
    chatbot = gr.Chatbot(label="Chat", height=400)
    msg_in = gr.Textbox(placeholder="Ask about Samsung Galaxy...", label="Question")
    submit_btn = gr.Button("Submit")

    def respond(message, history, model_choice):
        if not (message and message.strip()):
            return history, ""
        new_history, _ = chat_fn(message, history, model_choice)
        return new_history, ""

    msg_in.submit(respond, inputs=[msg_in, chatbot, model_dropdown], outputs=[chatbot, msg_in])
    submit_btn.click(respond, inputs=[msg_in, chatbot, model_dropdown], outputs=[chatbot, msg_in])
    gr.Examples(
        examples=["What is the battery size of the S24 Ultra?", "Does the Z Flip 6 have wireless charging?", "Which A series has IP67?"],
        inputs=msg_in,
        label="Example questions",
    )

demo.launch()